# Deep Learning 2025 — Homework Week 7
## Sequence & Language Models: N-grams, Markov Models, and a small RNN

Please fill the `TODO` blocks.

In [2]:
# imports and global settings
import math
import random
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

# Exercise 1 — N-gram Language Models & Zipf’s Law

We’ll work on a tiny toy corpus and:
1. Tokenize and count word frequencies
2. Plot word rank vs frequency (Zipf’s law)
3. Implement unigram and bigram language models with Laplace smoothing
4. Compare sentence probabilities & perplexities

### 1.1 Toy corpus and tokenization

In [3]:
corpus = """
Deep learning is fun and powerful .
Deep learning models can learn complex patterns .
Language models estimate probabilities over sequences of words .
We can build tiny language models with unigrams , bigrams , and trigrams .
"""

# very simple tokenizer: lowercase + split on spaces + keep punctuation tokens
def tokenize(text):
    tokens = text.lower().strip().split()
    return tokens

tokens = tokenize(corpus)
print(tokens[:30])
print("Number of tokens:", len(tokens))

['deep', 'learning', 'is', 'fun', 'and', 'powerful', '.', 'deep', 'learning', 'models', 'can', 'learn', 'complex', 'patterns', '.', 'language', 'models', 'estimate', 'probabilities', 'over', 'sequences', 'of', 'words', '.', 'we', 'can', 'build', 'tiny', 'language', 'models']
Number of tokens: 38


Build a vocabulary and count token frequencies.

In [ ]:
# TODO: build vocabulary and counts
# Hint: use collections.Counter

# counts = ...
# vocab = sorted(counts.keys())
# word2id = {w:i for i,w in enumerate(vocab)}
# id2word = {i:w for w,i in word2id.items()}

print("Vocab size:", len(vocab))
print("Most common 10:", counts.most_common(10))

---

### 1.2 Zipf's Law
Plot rank vs frequency on a log-log scale and comment on the shape.


In [ ]:
# sort words by frequency (descending)
sorted_counts = counts.most_common()
ranks = np.arange(1, len(sorted_counts)+1)
freqs = np.array([c for _, c in sorted_counts])

plt.loglog(ranks, freqs, marker="o")
plt.xlabel("Rank (log)")
plt.ylabel("Frequency (log)")
plt.title("Zipf-like behavior on tiny corpus")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.show()

**Reflection (no need to write an answer):**
- Does the plot look roughly like a straight line on a log-log scale?
- What would happen to the plot if we used a *much* larger corpus?

---

### 1.3 Unigram Language Model with Laplace Smoothing
We estimate:
$ P(w) = (count(w) + alpha) / (N + alpha * |V|) $
where:
  - N is total token count
  - |V| is vocabulary size
  - alpha > 0 is the Laplace smoothing parameter

We'll write functions to compute sentence log-probability and perplexity.


In [ ]:
N = sum(counts.values())
V = len(vocab)

def unigram_prob(word, counts, N, V, alpha=1.0):
    # TODO: implement Laplace-smoothed unigram probability
    # p(w) = (count(w) + alpha) / (N + alpha * V)
    raise NotImplementedError

def sentence_log_prob_unigram(sentence_tokens, counts, N, V, alpha=1.0):
    logp = 0.0
    for w in sentence_tokens:
        p = unigram_prob(w, counts, N, V, alpha)
        logp += math.log(p)
    return logp

def perplexity_from_logp(logp, T):
    # T = number of tokens in the sentence
    return math.exp(-logp / T)

# test sentences
sent1 = tokenize("deep learning models are powerful .")
sent2 = tokenize("trigrams with deep language sequences .")

for s in [sent1, sent2]:
    lp = sentence_log_prob_unigram(s, counts, N, V, alpha=1.0)
    ppl = perplexity_from_logp(lp, len(s))
    print("Sentence:", " ".join(s))
    print("  log P =", lp, " | perplexity =", ppl)

---

### 1.4 Bigram Language Model with Laplace Smoothing

Now model:
$  P(w_t | w_{t-1}) = (count(w_{t-1}, w_t) + alpha) / (count(w_{t-1}) + alpha * |V|) $

We'll:
1. Count bigrams
2. Implement bigram conditional probabilities with Laplace smoothing
3. Compute sentence log-probabilities and compare to unigram

In [ ]:
# TODO: build bigram count matrix or dict
# Option 1: 2D numpy array of shape (V, V)
# Option 2: dict[(prev, curr)] -> count

bigram_counts = np.zeros((V, V), dtype=int)

for w_prev, w_curr in zip(tokens[:-1], tokens[1:]):
    i = word2id[w_prev]
    j = word2id[w_curr]
    bigram_counts[i, j] += 1

print("Some bigram counts (as (prev, curr, count)):")
example_pairs = [("language", "models"), ("deep", "learning"), ("with", "unigrams")]
for w_prev, w_curr in example_pairs:
    i, j = word2id[w_prev], word2id[w_curr]
    print(w_prev, "->", w_curr, ":", bigram_counts[i, j])

In [ ]:
def bigram_prob(curr_word, prev_word, counts_unigram, bigram_counts, V, alpha=1.0):
    # TODO: compute P(curr | prev) with Laplace smoothing
    # numerator = count(prev, curr) + alpha
    # denominator = count(prev) + alpha * V
    raise NotImplementedError

def sentence_log_prob_bigram(sentence_tokens, counts_unigram, bigram_counts, V, alpha=1.0):
    # assume first token has no context; you can use unigram prob for it
    if len(sentence_tokens) == 0:
        return 0.0
    logp = math.log(unigram_prob(sentence_tokens[0], counts_unigram, N, V, alpha))
    for prev, curr in zip(sentence_tokens[:-1], sentence_tokens[1:]):
        p = bigram_prob(curr, prev, counts_unigram, bigram_counts, V, alpha)
        logp += math.log(p)
    return logp

for s in [sent1, sent2]:
    lp_uni = sentence_log_prob_unigram(s, counts, N, V, alpha=1.0)
    lp_bi = sentence_log_prob_bigram(s, counts, bigram_counts, V, alpha=1.0)
    ppl_uni = perplexity_from_logp(lp_uni, len(s))
    ppl_bi = perplexity_from_logp(lp_bi, len(s))
    print("Sentence:", " ".join(s))
    print("  Unigram perplexity:", ppl_uni)
    print("  Bigram  perplexity:", ppl_bi)

**Reflection:**
- For these sentences, does the bigram model generally assign **lower perplexity** than the unigram model?  
- Why might a bigram (or trigram) model be better for capturing language structure?

---

# Exercise 2 — Markov Models for Sequences

Here we’ll use a discrete Markov chain to model a simple “weather” process.
1. Estimate a first-order transition matrix $ P(state_t | state_{t-1}) $ from data
2. Sample sequences from the model
3. Extend to a second-order Markov model and compare

### 2.1 Synthetic weather sequence and first-order Markov chain

In [ ]:
states = ["sunny", "cloudy", "rainy"]
state2id = {s:i for i,s in enumerate(states)}
id2state = {i:s for s,i in state2id.items()}

# We'll create a synthetic sequence from a "true" transition matrix,
# but you can imagine this comes from real data.
true_T = np.array([
    [0.7, 0.2, 0.1],  # from sunny
    [0.3, 0.4, 0.3],  # from cloudy
    [0.2, 0.5, 0.3],  # from rainy
])

def sample_markov(T, start_state_idx, length=50):
    seq = [start_state_idx]
    for _ in range(length-1):
        curr = seq[-1]
        next_state = np.random.choice(len(states), p=T[curr])
        seq.append(next_state)
    return seq

seq_idx = sample_markov(true_T, start_state_idx=0, length=200)
seq_states = [id2state[i] for i in seq_idx]
print("First 30 states:", seq_states[:30])

---

### 2.2 Estimate transition matrix from data (first-order)
Estimate $ \hat{P}(s_t | s_{t-1}) $ by counting transitions.


In [ ]:
# TODO: estimate transition counts and probabilities from seq_idx

num_states = len(states)
count_T = np.zeros((num_states, num_states), dtype=int)

for i_prev, i_curr in zip(seq_idx[:-1], seq_idx[1:]):
    # TODO: increment count from i_prev to i_curr
    pass

# convert counts to probabilities (row-normalized)
est_T = count_T / count_T.sum(axis=1, keepdims=True)

print("Estimated transition matrix:")
print(est_T)
print("True transition matrix:")
print(true_T)

---

### 2.3 Sample from the estimated chain and compare qualitatively

In [ ]:
est_seq_idx = sample_markov(est_T, start_state_idx=0, length=50)
est_seq_states = [id2state[i] for i in est_seq_idx]
print("Sample from estimated chain:")
print(est_seq_states)

**Reflection:**
- How close is the estimated transition matrix to the true one?  
- If you had a much shorter sequence (e.g., length 20), what would happen?

---

### 2.4 Second-order Markov model $ (P(s_t | s_{t-1}, s_{t-2})) $
For small state spaces, we can estimate
$    P(s_t | s_{t-1}, s_{t-2}) $
by counting triples $ (s_{t-2}, s_{t-1}, s_t). $

We'll compare these conditional probabilities for a chosen pair $ (s_{t-2}, s_{t-1}) $
to the first-order model $ P(s_t | s_{t-1}) $.

In [ ]:
# TODO: estimate second-order counts and probabilities
second_order_counts = defaultdict(int)
pair_counts = defaultdict(int)

for a, b, c in zip(seq_idx[:-2], seq_idx[1:-1], seq_idx[2:]):
    # triple (a,b,c)
    pair_counts[(a,b)] += 1
    second_order_counts[(a,b,c)] += 1

def second_order_cond_probs(a, b):
    # returns P(s_t | a,b) over all possible c
    total = pair_counts[(a,b)]
    if total == 0:
        return np.ones(num_states) / num_states  # fallback uniform
    probs = np.zeros(num_states)
    for c in range(num_states):
        probs[c] = second_order_counts[(a,b,c)] / total
    return probs

# pick a specific pair, e.g. (sunny, cloudy)
a, b = state2id["sunny"], state2id["cloudy"]
p_second = second_order_cond_probs(a, b)
p_first = est_T[b]

print("P(s_t | sunny, cloudy) (second-order est):", p_second)
print("P(s_t | cloudy)        (first-order est) :", p_first)
print("States order:", states)

**Reflection:**
- In what situation can a **second-order** model capture structure that a first-order model cannot?  
- How is this conceptually related to **bigrams/trigrams** in language modeling?

---

# Exercise 3 — Tiny Character-Level RNN Language Model (PyTorch)

Here we connect the latent autoregressive / RNN idea to code.

We’ll:
1. Use the same corpus from Exercise 1
2. Build a small character vocabulary
3. Train a 1-layer GRU to predict the next character
4. Sample a few characters from the trained model


### 3.1 Setup: character vocabulary & dataset

In [ ]:
import torch
from torch import nn, optim

torch.manual_seed(42)

text = corpus.lower()
chars = sorted(list(set(text)))
print("Chars:", chars)
vocab_size = len(chars)
char2id = {ch:i for i,ch in enumerate(chars)}
id2char = {i:ch for ch,i in char2id.items()}

def encode(text):
    return [char2id[ch] for ch in text]

def decode(indices):
    return "".join(id2char[i] for i in indices)

encoded = encode(text)
print("Encoded length:", len(encoded))

We'll build training samples of fixed length `seq_len`.

Each sample is a sequence of input characters and the target is the shifted sequence (next char at each step).


In [ ]:
seq_len = 40

def make_sequences(encoded, seq_len):
    X, Y = [], []
    for i in range(len(encoded) - seq_len):
        x_seq = encoded[i : i + seq_len]
        y_seq = encoded[i+1 : i + seq_len + 1]
        X.append(x_seq)
        Y.append(y_seq)
    return np.array(X, dtype=np.int64), np.array(Y, dtype=np.int64)

X_seqs, Y_seqs = make_sequences(encoded, seq_len)
print("Num sequences:", X_seqs.shape[0], "Seq length:", X_seqs.shape[1])

---

### 3.2 Model: simple GRU-based character LM (given)
We'll use an embedding, a GRU, and a linear layer to produce logits over characters.


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        # x: (batch, seq_len)
        emb = self.embed(x)                 # (batch, seq_len, emb_dim)
        out, h_next = self.gru(emb, h)      # (batch, seq_len, hidden_dim)
        logits = self.fc(out)               # (batch, seq_len, vocab_size)
        return logits, h_next

model = CharRNN(vocab_size)
criterion = nn.CrossEntropyLoss()  # note: expects (N, C) logits and (N,) targets
optimizer = optim.Adam(model.parameters(), lr=1e-2)

---

### 3.3 Training loop (TODO)
Implement a simple training loop:
- Use mini-batches
- For each batch:
  1. Forward pass: compute logits
  2. Reshape logits and targets so CrossEntropyLoss works
  3. Backward + optimizer step
- Record training loss per epoch

In [ ]:
def get_batches(X, Y, batch_size=32):
    idx = np.arange(len(X))
    np.random.shuffle(idx)
    for start in range(0, len(X), batch_size):
        batch_idx = idx[start:start+batch_size]
        yield X[batch_idx], Y[batch_idx]

num_epochs = 15
batch_size = 32
loss_history = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    n_batches = 0
    for Xb, Yb in get_batches(X_seqs, Y_seqs, batch_size=batch_size):
        Xb_t = torch.tensor(Xb, dtype=torch.long)
        Yb_t = torch.tensor(Yb, dtype=torch.long)

        # TODO:
        # 1) Zero gradients
        # 2) Forward pass: logits, _ = model(Xb_t)
        # 3) Reshape logits to (N, C) and Yb_t to (N,)
        # 4) Compute loss
        # 5) Backprop + step

        # epoch_loss += loss.item()
        # n_batches += 1
        pass

    # avg_loss = epoch_loss / n_batches
    # loss_history.append(avg_loss)
    # print(f"Epoch {epoch+1}/{num_epochs} - loss: {avg_loss:.4f}")

After you implement the training loop and run it, you should see the loss decreasing.

You can then plot the loss curve:


In [7]:
# import matplotlib.pyplot as plt
# plt.plot(loss_history)
# plt.xlabel("Epoch")
# plt.ylabel("Training loss")
# plt.title("Char-RNN training loss")
# plt.show()

---

### 3.4 Sampling from the trained model (optional)
Write a small function that:
- Starts from a given "seed" text
- Feeds characters through the model
- At each step, samples the next character from the output distribution

This part can be fairly free-form.

In [8]:
def sample_model(model, seed_text="deep ", length=100, temperature=1.0):
    model.eval()
    with torch.no_grad():
        input_ids = torch.tensor([[char2id[ch] for ch in seed_text]], dtype=torch.long)
        h = None
        generated = list(seed_text)
        for _ in range(length):
            logits, h = model(input_ids, h)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            generated.append(id2char[next_id])
            input_ids = torch.tensor([[next_id]], dtype=torch.long)
    return "".join(generated)

# After training:
# print(sample_model(model, seed_text="language ", length=120, temperature=0.8))

**Reflection:**
- In what sense is this RNN a **latent autoregressive model**?  
- How does it relate conceptually to the **n-gram** models from Exercise 1?